# Demo happy path - IO flows
### `Import trips → Validate trips → Build flows → Write flows → Read flows`

En este notebook quiero probar un flujo completo y exitoso del módulo usando un dataset sintético pensado específicamente para construir flujos útiles.

La idea no es forzar casos problemáticos, sino verificar que:
- el import deja un `TripDataset` bien formado,
- la validación formal pasa correctamente,
- la construcción de flows genera resultados suficientemente ricos,
- la persistencia formal de flows funciona,
- y la lectura reconstruye un `FlowDataset` utilizable.

### Configuración inicial

In [3]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic"

### 0. Setup e imports

In [4]:
from pathlib import Path
import json
import pandas as pd

from pylondrina.importing import import_trips_from_dataframe, ImportOptions
from pylondrina.validation import validate_trips, ValidationOptions
from pylondrina.transforms.flows import build_flows, FlowBuildOptions
from pylondrina.io.flows import write_flows, read_flows, WriteFlowsOptions, ReadFlowsOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec


def issues_to_frame(issues):
    rows = []
    for issue in issues:
        rows.append(
            {
                "level": getattr(issue, "level", None),
                "code": getattr(issue, "code", None),
                "message": getattr(issue, "message", None),
                "field": getattr(issue, "field", None),
                "source_field": getattr(issue, "source_field", None),
                "row_count": getattr(issue, "row_count", None),
                "details": getattr(issue, "details", None),
            }
        )
    return pd.DataFrame(rows)


def sort_df(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    return df.sort_values(by=by).reset_index(drop=True)

SOURCE_CSV = DATA_PATH / "demo_trips_for_flows_hp.csv"
OUTPUT_DIR = DATA_PATH / "demo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FLOWS_ARTIFACT_BASE = OUTPUT_DIR / "demo_flows_happy_path"

print("SOURCE_CSV =", SOURCE_CSV.resolve())
print("FLOWS_ARTIFACT_BASE =", FLOWS_ARTIFACT_BASE.resolve())

SOURCE_CSV = D:\Memoria\repos\pylondrina\data\synthetic\demo_trips_for_flows_hp.csv
FLOWS_ARTIFACT_BASE = D:\Memoria\repos\pylondrina\data\synthetic\demo_outputs\demo_flows_happy_path


## 1. Cargo el CSV fuente y reviso su estructura

Primero quiero confirmar que el dataset fuente tiene el tamaño y la riqueza esperada antes de entrar al pipeline del módulo.

In [5]:
df_source = pd.read_csv(SOURCE_CSV)

print("shape fuente:", df_source.shape)
print("n columnas fuente:", len(df_source.columns))
display(df_source.head(5))
display(pd.Series(df_source.columns, name="source_columns").to_frame())

shape fuente: (15000, 42)
n columnas fuente: 42


,id_movimiento,id_usuario,id_viaje,orden_movimiento,lon_origen,lat_origen,lon_destino,lat_destino,ts_salida_utc,ts_llegada_utc,...,codigo_ruta_tp,distancia_ruta_m,ingreso_hogar_clp,ingreso_personal_clp,estado_actividad,tipo_ocupacion,job_sector,work_schedule,travel_time_bucket,income_imputed_flag
0,m00000,u0135,t00000,0,-70.769604,-33.502249,-70.681907,-33.446320,2026-03-04T11:47:00Z,2026-03-04T12:55:00Z,...,I09c,8905.0,2326677.0,1205840.0,retired,informal,education,flexible,41-60,not_imputed
1,m00001,u4536,t00001,0,-70.745590,-33.520438,-70.666115,-33.454851,2026-03-02T21:24:00Z,2026-03-02T22:52:00Z,...,L1,9914.0,2221169.0,609863.0,homemaker,self_employed,education,shift,11-20,not_imputed
2,m00002,u6282,t00002,0,-70.731247,-33.361678,-70.667184,-33.438239,2026-03-06T15:03:00Z,2026-03-06T16:51:00Z,...,I09c,11065.3,634080.0,1623388.0,retired,public_sector,health,night_shift,0-10,imputed_missing_band
3,m00003,u6043,t00003,0,-70.593399,-33.443707,-70.609342,-33.433586,2026-03-02T04:20:00Z,2026-03-02T05:08:00Z,...,L1,11309.9,1327818.0,1055120.0,retired,public_sector,services,night_shift,60+,fully_imputed
4,m00004,u1170,t00004,0,-70.726012,-33.348403,-70.683511,-33.448064,2026-03-02T12:20:00Z,2026-03-02T12:35:00Z,...,401,8896.4,1118444.0,1777696.0,other,self_employed,services,full_time,60+,fully_imputed


,source_columns
0,id_movimiento
1,id_usuario
2,id_viaje
3,orden_movimiento
4,lon_origen
5,lat_origen
6,lon_destino
7,lat_destino
8,ts_salida_utc
9,ts_llegada_utc


## 2. Defino el contrato de importación y validación

En esta demo quiero que el source dataset tenga nombres de campos “externos”, pero que el dominio semántico ya venga limpio para mantener el flujo en modo happy path.

Por eso:
- uso una correspondencia de campos explícita,
- dejo `value_correspondence = {}` a propósito,
- y construyo un `TripSchema` acotado, pero suficientemente exigente.

In [12]:
FIELD_CORRESPONDENCE_DEMO = {
    "movement_id": "id_movimiento",
    "user_id": "id_usuario",
    "origin_longitude": "lon_origen",
    "origin_latitude": "lat_origen",
    "destination_longitude": "lon_destino",
    "destination_latitude": "lat_destino",
    "origin_time_utc": "ts_salida_utc",
    "destination_time_utc": "ts_llegada_utc",
    "trip_id": "id_viaje",
    "movement_seq": "orden_movimiento",
    "origin_municipality": "comuna_origen",
    "destination_municipality": "comuna_destino",
    "timezone_offset_min": "utc_offset_min",
    "trip_weight": "factor_expansion",
    "mode_sequence": "secuencia_modos",
    "mode": "modo",
    "purpose": "proposito",
    "day_type": "tipo_dia",
    "time_period": "periodo_horario",
    "user_gender": "genero",
    "user_age_group": "tramo_edad",
    "income_quintile": "quintil_ingreso",
    "travel_time_min": "tiempo_viaje_min",
    "fare_amount": "tarifa_clp",
    "public_transport_route_code": "codigo_ruta_tp",
    "distance_route_m": "distancia_ruta_m",
    "household_income_clp": "ingreso_hogar_clp",
    "personal_income_clp": "ingreso_personal_clp",
    "activity_status": "estado_actividad",
    "occupation_type": "tipo_ocupacion",
}

# En esta demo lo dejo vacío a propósito para que la validación sea un happy path puro.
VALUE_CORRESPONDENCE_DEMO = {}

MODE_VALUES = [
    "walk", "bicycle", "scooter", "motorcycle", "car",
    "taxi", "ride_hailing", "bus", "metro", "train", "other",
]
PURPOSE_VALUES = [
    "home", "work", "education", "shopping", "errand",
    "health", "leisure", "transfer", "other",
]
DAY_TYPE_VALUES = ["weekday", "weekend", "holiday"]
TIME_PERIOD_VALUES = ["night", "morning", "midday", "afternoon", "evening"]
USER_GENDER_VALUES = ["female", "male", "other", "unknown"]
USER_AGE_VALUES = ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"]
INCOME_Q_VALUES = ["1", "2", "3", "4", "5", "unknown"]
ACTIVITY_STATUS_VALUES = ["working", "studying", "homemaker", "retired", "unemployed", "other"]
OCCUPATION_TYPE_VALUES = ["employee", "self_employed", "employer", "public_sector", "informal"]

required_fields = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_h3_index",
    "destination_h3_index",
    "origin_time_utc",
    "destination_time_utc",
    "trip_id",
    "movement_seq",
]

trip_fields = {
    "movement_id": FieldSpec(
        name="movement_id",
        dtype="string",
        required=True,
        constraints={"nullable": False, "unique": True},
    ),
    "user_id": FieldSpec(
        name="user_id",
        dtype="string",
        required=True,
        constraints={"nullable": False},
    ),
    "origin_longitude": FieldSpec(
        name="origin_longitude",
        dtype="float",
        required=True,
        constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
    ),
    "origin_latitude": FieldSpec(
        name="origin_latitude",
        dtype="float",
        required=True,
        constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
    ),
    "destination_longitude": FieldSpec(
        name="destination_longitude",
        dtype="float",
        required=True,
        constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
    ),
    "destination_latitude": FieldSpec(
        name="destination_latitude",
        dtype="float",
        required=True,
        constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
    ),
    "origin_h3_index": FieldSpec(
        name="origin_h3_index",
        dtype="string",
        required=True,
        constraints={"nullable": False, "h3": {"require_valid": True, "resolution": 12}},
    ),
    "destination_h3_index": FieldSpec(
        name="destination_h3_index",
        dtype="string",
        required=True,
        constraints={"nullable": False, "h3": {"require_valid": True, "resolution": 12}},
    ),
    "origin_time_utc": FieldSpec(
        name="origin_time_utc",
        dtype="datetime",
        required=True,
        constraints={"nullable": False, "datetime": {"allow_naive": False, "timezone": "UTC"}},
    ),
    "destination_time_utc": FieldSpec(
        name="destination_time_utc",
        dtype="datetime",
        required=True,
        constraints={"nullable": False, "datetime": {"allow_naive": False, "timezone": "UTC"}},
    ),
    "trip_id": FieldSpec(
        name="trip_id",
        dtype="string",
        required=True,
        constraints={"nullable": False},
    ),
    "movement_seq": FieldSpec(
        name="movement_seq",
        dtype="int",
        required=True,
        constraints={"nullable": False, "range": {"min": 0}},
    ),
    "origin_municipality": FieldSpec(
        name="origin_municipality",
        dtype="string",
        required=False,
        constraints={"nullable": True},
    ),
    "destination_municipality": FieldSpec(
        name="destination_municipality",
        dtype="string",
        required=False,
        constraints={"nullable": True},
    ),
    "timezone_offset_min": FieldSpec(
        name="timezone_offset_min",
        dtype="int",
        required=False,
        constraints={"nullable": True, "range": {"min": -720, "max": 840}},
    ),
    "trip_weight": FieldSpec(
        name="trip_weight",
        dtype="float",
        required=False,
        constraints={"nullable": True, "range": {"min": 0.0}},
    ),
    "mode_sequence": FieldSpec(
        name="mode_sequence",
        dtype="string",
        required=False,
        constraints={"nullable": True},
    ),
    "mode": FieldSpec(
        name="mode",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=MODE_VALUES, extendable=False),
    ),
    "purpose": FieldSpec(
        name="purpose",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=PURPOSE_VALUES, extendable=False),
    ),
    "day_type": FieldSpec(
        name="day_type",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=DAY_TYPE_VALUES, extendable=False),
    ),
    "time_period": FieldSpec(
        name="time_period",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=TIME_PERIOD_VALUES, extendable=False),
    ),
    "user_gender": FieldSpec(
        name="user_gender",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=USER_GENDER_VALUES, extendable=False),
    ),
    "user_age_group": FieldSpec(
        name="user_age_group",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=USER_AGE_VALUES, extendable=False),
    ),
    "income_quintile": FieldSpec(
        name="income_quintile",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=INCOME_Q_VALUES, extendable=False),
    ),
    "travel_time_min": FieldSpec(
        name="travel_time_min",
        dtype="float",
        required=False,
        constraints={"nullable": True, "range": {"min": 0.0}},
    ),
    "fare_amount": FieldSpec(
        name="fare_amount",
        dtype="float",
        required=False,
        constraints={"nullable": True, "range": {"min": 0.0}},
    ),
    "public_transport_route_code": FieldSpec(
        name="public_transport_route_code",
        dtype="string",
        required=False,
        constraints={"nullable": True},
    ),
    "distance_route_m": FieldSpec(
        name="distance_route_m",
        dtype="float",
        required=False,
        constraints={"nullable": True, "range": {"min": 0.0}},
    ),
    "household_income_clp": FieldSpec(
        name="household_income_clp",
        dtype="float",
        required=False,
        constraints={"nullable": True, "range": {"min": 0.0}},
    ),
    "personal_income_clp": FieldSpec(
        name="personal_income_clp",
        dtype="float",
        required=False,
        constraints={"nullable": True, "range": {"min": 0.0}},
    ),
    "activity_status": FieldSpec(
        name="activity_status",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=ACTIVITY_STATUS_VALUES, extendable=False),
    ),
    "occupation_type": FieldSpec(
        name="occupation_type",
        dtype="categorical",
        required=False,
        constraints={"nullable": True},
        domain=DomainSpec(values=OCCUPATION_TYPE_VALUES, extendable=False),
    ),
}

trip_schema_demo = TripSchema(
    version="1.1",
    fields=trip_fields,
    required=required_fields,
)

provenance_demo = {
    "source": {
        "name": "synthetic_demo_flows_source",
        "entity": "trips",
    },
    "coverage": {
        "spatial": {"city": "Santiago"},
        "temporal": {"kind": "synthetic_week"},
    },
    "notes": {
        "purpose": "happy_path_demo_for_flows_persistence"
    },
}

print("n campos schema:", len(trip_schema_demo.fields))
print("required:", trip_schema_demo.required)

n campos schema: 32
required: ['movement_id', 'user_id', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_h3_index', 'destination_h3_index', 'origin_time_utc', 'destination_time_utc', 'trip_id', 'movement_seq']


## 3. Importo el dataset al formato Golondrina

Aquí quiero comprobar tres cosas:
1. que el import funciona con correspondencia de campos,
2. que derive correctamente H3 a resolución 12,
3. y que el dataset quede rico, no solo “válido”.

In [13]:
import_options_demo = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone=None,
)

trips, import_report = import_trips_from_dataframe(
    df_source,
    trip_schema_demo,
    source_name="synthetic_demo_flows_source",
    options=import_options_demo,
    field_correspondence=FIELD_CORRESPONDENCE_DEMO,
    value_correspondence=VALUE_CORRESPONDENCE_DEMO,
    provenance=provenance_demo,
    h3_resolution=12,
)

print("shape trips importados:", trips.data.shape)
print("n columnas trips importados:", len(trips.data.columns))
display(import_report.summary)
display(issues_to_frame(import_report.issues).head(20))
display(
    trips.data[
        [
            "movement_id",
            "origin_h3_index",
            "destination_h3_index",
            "origin_time_utc",
            "destination_time_utc",
            "mode",
            "day_type",
            "trip_weight",
        ]
    ].head(5)
)

print("metadata['is_validated'] =", trips.metadata.get("is_validated"))
print("metadata temporal =", trips.metadata.get("temporal"))

shape trips importados: (15000, 44)
n columnas trips importados: 44


{'rows_in': 15000,
 'rows_out': 15000,
 'n_fields_mapped': 30,
 'n_domain_mappings_applied': 0}

,level,code,message,field,source_field,row_count,details
0,info,IMP.METADATA.DATASET_ID_CREATED,Se generó dataset_id para el dataset importado...,None,None,None,{'dataset_id': 'tripds_893956e947fa445292e5003...


,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,day_type,trip_weight
0,m00000,8cb2c542b9813ff,8cb2c5543c5a1ff,2026-03-04 11:47:00+00:00,2026-03-04 12:55:00+00:00,scooter,holiday,3.688
1,m00001,8cb2c5476a923ff,8cb2c554ec803ff,2026-03-02 21:24:00+00:00,2026-03-02 22:52:00+00:00,metro,weekday,3.164
2,m00002,8cb2c552906e5ff,8cb2c55415a1dff,2026-03-06 15:03:00+00:00,2026-03-06 16:51:00+00:00,ride_hailing,holiday,2.395
3,m00003,8cb2c50b6c411ff,8cb2c556b8f37ff,2026-03-02 04:20:00+00:00,2026-03-02 05:08:00+00:00,walk,weekend,1.120
4,m00004,8cb2c552d2535ff,8cb2c5543565dff,2026-03-02 12:20:00+00:00,2026-03-02 12:35:00+00:00,car,weekend,2.713


metadata['is_validated'] = False
metadata temporal = {'tier': 'tier_1', 'fields_present': ['origin_time_utc', 'destination_time_utc'], 'normalization': {'origin_time_utc': {'status': 'string_tzaware_to_utc', 'tz_kind': 'none', 'n_nat': 0}, 'destination_time_utc': {'status': 'string_tzaware_to_utc', 'tz_kind': 'none', 'n_nat': 0}}}


## 4. Valido el `TripDataset`

En esta demo sí quiero correr una validación bastante completa:
- required fields,
- tipos y formatos,
- constraints,
- dominios completos,
- consistencia temporal,
- y duplicados por `movement_id`.

La idea es que si aquí pasa, ya puedo tratar el dataset como una base limpia para construir flows.

In [14]:
validation_options_demo = ValidationOptions(
    strict=False,
    validate_required_fields=True,
    validate_types_and_formats=True,
    validate_constraints=True,
    validate_domains="full",
    validate_temporal_consistency=True,
    validate_duplicates=True,
    duplicates_subset=("movement_id",),
    allow_partial_od_spatial=False,
)

validation_report = validate_trips(
    trips,
    options=validation_options_demo,
)

display(validation_report.summary)
display(issues_to_frame(validation_report.issues).head(20))
display(validation_report.issues)

assert validation_report.ok is True
assert trips.metadata.get("is_validated") is True

print("Validation ok =", validation_report.ok)
print("metadata['is_validated'] =", trips.metadata.get("is_validated"))

{'ok': True,
 'n_rows': 15000,
 'n_issues': 0,
 'n_errors': 0,
 'n_warnings': 0,
 'n_info': 0,
 'counts_by_level': {'error': 0, 'warning': 0, 'info': 0},
 'counts_by_code': {},
 'checked_fields': ['activity_status',
  'day_type',
  'destination_h3_index',
  'destination_latitude',
  'destination_longitude',
  'destination_municipality',
  'destination_time_utc',
  'distance_route_m',
  'fare_amount',
  'household_income_clp',
  'income_quintile',
  'mode',
  'mode_sequence',
  'movement_id',
  'movement_seq',
  'occupation_type',
  'origin_h3_index',
  'origin_latitude',
  'origin_longitude',
  'origin_municipality',
  'origin_time_utc',
  'personal_income_clp',
  'public_transport_route_code',
  'purpose',
  'time_period',
  'timezone_offset_min',
  'travel_time_min',
  'trip_id',
  'trip_weight',
  'user_age_group',
  'user_gender',
  'user_id'],
 'checks_executed': {'required_fields': True,
  'types_and_formats': True,
  'constraints': True,
  'domains': True,
  'temporal_consistenc

""


[]

Validation ok = True
metadata['is_validated'] = True


## 5. Construyo flows desde trips

Aquí ya bajo la resolución espacial a H3 7, para obtener una agregación más útil visual y analíticamente.

Para no dispersar demasiado los flows:
- segmento solo por `mode`,
- agrego temporalmente por `day`,
- y uso un umbral mínimo de viajes por flow.

In [27]:
build_options_demo = FlowBuildOptions(
    h3_resolution=7,
    group_by=["mode"],
    time_aggregation="day",
    time_basis="origin",
    min_trips_per_flow=4,
    keep_flow_to_trips=True,
    require_validated=True,
    strict=False,
)

flows, flow_build_report = build_flows(
    trips,
    options=build_options_demo,
)

display(flow_build_report.summary)
display(issues_to_frame(flow_build_report.issues).head(20))

print("shape flows:", flows.flows.shape)
print("shape flow_to_trips:", None if flows.flow_to_trips is None else flows.flow_to_trips.shape)
print("aggregation_spec:")
display(flows.aggregation_spec)

display(
    flows.flows.sort_values(["flow_count", "flow_value"], ascending=[False, False]).head(15)
)

display(
    flows.flows.groupby("mode", dropna=False)["flow_count"].agg(["count", "sum"]).sort_values("sum", ascending=False)
)

{'n_trips_in': 15000,
 'n_trips_eligible': 15000,
 'n_trips_dropped': 0,
 'n_flows_out': 840,
 'n_flow_to_trips_rows': 4271}

""


shape flows: (840, 8)
shape flow_to_trips: (4271, 2)
aggregation_spec:


{'h3_resolution': 7,
 'group_by': ['mode'],
 'time_aggregation': 'day',
 'time_basis': 'origin',
 'min_trips_per_flow': 4,
 'keep_flow_to_trips': True,
 'require_validated': True,
 'strict': False,
 'max_issues': 1000,
 'effective_flow_keys': ['origin_h3_index',
  'destination_h3_index',
  'window_start_utc',
  'window_end_utc',
  'mode']}

,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,flow_count,flow_value
339,flow_0000339,87b2c542bffffff,87b2c5543ffffff,2026-03-06,2026-03-07,scooter,13,34.132
657,flow_0000657,87b2c555cffffff,87b2c519effffff,2026-03-01,2026-03-02,bus,12,29.923
646,flow_0000646,87b2c555cffffff,87b2c519affffff,2026-03-06,2026-03-07,ride_hailing,11,27.415
334,flow_0000334,87b2c542bffffff,87b2c5543ffffff,2026-03-06,2026-03-07,bus,10,33.056
66,flow_0000066,87b2c5093ffffff,87b2c556bffffff,2026-03-04,2026-03-05,motorcycle,10,31.616
215,flow_0000215,87b2c5425ffffff,87b2c5543ffffff,2026-03-06,2026-03-07,walk,10,28.737
295,flow_0000295,87b2c542bffffff,87b2c5543ffffff,2026-03-01,2026-03-02,taxi,10,27.141
323,flow_0000323,87b2c542bffffff,87b2c5543ffffff,2026-03-04,2026-03-05,scooter,10,25.800
631,flow_0000631,87b2c555cffffff,87b2c519affffff,2026-03-04,2026-03-05,other,10,17.820
719,flow_0000719,87b2c5731ffffff,87b2c5543ffffff,2026-03-02,2026-03-03,bicycle,10,17.664


,count,sum
mode,,
bicycle,87,434
train,83,429
taxi,83,421
car,83,420
bus,77,403
walk,77,390
motorcycle,77,378
scooter,70,367
ride_hailing,68,357


## 6. Persisto el `FlowDataset`

Ahora quiero dejar el artefacto formal de flows en disco para comprobar que:
- se escriba el parquet principal,
- se escriba el sidecar,
- se persista también `flow_to_trips`,
- y quede un bundle Golondrina reutilizable.

In [28]:
write_options_demo = WriteFlowsOptions(
    mode="overwrite",
    storage_format="parquet",
    parquet_compression="snappy",
    normalize_artifact_dir=True,
    write_flow_to_trips=True,
)

write_report = write_flows(
    flows,
    FLOWS_ARTIFACT_BASE,
    options=write_options_demo,
)

artifact_root = Path(str(FLOWS_ARTIFACT_BASE) + ".golondrina")

display(write_report.summary)
display(issues_to_frame(write_report.issues).head(20))

print("artifact_root =", artifact_root.resolve())
print("exists:", artifact_root.exists())
print("files:")
for path in sorted(artifact_root.iterdir()):
    print(" -", path.name, f"({path.stat().st_size} bytes)")

{'n_flows': 840,
 'n_flow_to_trips': 4271,
 'files_written': ['flows.parquet',
  'flows.metadata.json',
  'flow_to_trips.parquet'],
 'dataset_id': 'flows_27dedf8e1df4',
 'artifact_id': 'art_f2350103-946d-49f9-9328-f1760ad597d2',
 'path': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\demo_flows_happy_path.golondrina'}

""


artifact_root = D:\Memoria\repos\pylondrina\data\synthetic\demo_outputs\demo_flows_happy_path.golondrina
exists: True
files:
 - flow_to_trips.parquet (37747 bytes)
 - flows.metadata.json (9226 bytes)
 - flows.parquet (18620 bytes)


## 7. Leo el artefacto persistido

En esta última etapa quiero reconstruir el `FlowDataset` desde disco y revisar:
- que la tabla de flows vuelva correctamente,
- que vuelva también `flow_to_trips`,
- que `aggregation_spec` y `provenance` se conserven,
- que `source_trips` no se reconstruya,
- y que `metadata["is_validated"]` quede en `False` después del read.

In [29]:
read_options_demo = ReadFlowsOptions(
    strict=False,
    keep_metadata=True,
    read_flow_to_trips=True,
)

flows_loaded, read_report = read_flows(
    FLOWS_ARTIFACT_BASE,   # lo paso sin sufijo para probar también el fallback
    options=read_options_demo,
)

display(read_report.summary)
display(issues_to_frame(read_report.issues).head(20))

print("shape loaded flows:", flows_loaded.flows.shape)
print("shape loaded flow_to_trips:", None if flows_loaded.flow_to_trips is None else flows_loaded.flow_to_trips.shape)
print("loaded metadata['is_validated'] =", flows_loaded.metadata.get("is_validated"))
print("loaded source_trips =", flows_loaded.source_trips)

display(flows_loaded.flows.head(10))

{'n_flows': 840,
 'n_columns': 8,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 4271,
 'files_read': ['flows.parquet',
  'flow_to_trips.parquet',
  'flows.metadata.json'],
 'dataset_id': 'flows_27dedf8e1df4',
 'artifact_id': 'art_f2350103-946d-49f9-9328-f1760ad597d2'}

""


shape loaded flows: (840, 8)
shape loaded flow_to_trips: (4271, 2)
loaded metadata['is_validated'] = False
loaded source_trips = None


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,flow_count,flow_value
0,flow_0000000,87b2c5090ffffff,87b2c556bffffff,2026-03-02,2026-03-03,taxi,5,15.704
1,flow_0000001,87b2c5092ffffff,87b2c5568ffffff,2026-03-03,2026-03-04,walk,5,17.595
2,flow_0000002,87b2c5092ffffff,87b2c556affffff,2026-03-03,2026-03-04,car,4,14.363
3,flow_0000003,87b2c5092ffffff,87b2c556affffff,2026-03-04,2026-03-05,bus,4,13.513
4,flow_0000004,87b2c5092ffffff,87b2c556affffff,2026-03-04,2026-03-05,ride_hailing,4,10.181
5,flow_0000005,87b2c5092ffffff,87b2c556affffff,2026-03-05,2026-03-06,bus,5,19.656
6,flow_0000006,87b2c5092ffffff,87b2c556bffffff,2026-03-02,2026-03-03,other,4,8.469
7,flow_0000007,87b2c5092ffffff,87b2c556bffffff,2026-03-02,2026-03-03,taxi,4,8.817
8,flow_0000008,87b2c5092ffffff,87b2c556bffffff,2026-03-03,2026-03-04,other,4,14.498
9,flow_0000009,87b2c5092ffffff,87b2c556bffffff,2026-03-03,2026-03-04,train,4,9.446


## 8. Comparo el `FlowDataset` original con el reconstruido

No espero igualdad total de `metadata`, porque:
- `read_flows` agrega su propio evento,
- y además fuerza `metadata["is_validated"] = False`.

Pero sí quiero comprobar que se conservan bien:
- la tabla principal de flows,
- la tabla auxiliar `flow_to_trips`,
- `aggregation_spec`,
- `provenance`,
- `dataset_id`,
- y `artifact_id`.

In [30]:
pd.testing.assert_frame_equal(
    sort_df(flows.flows, ["flow_id"]),
    sort_df(flows_loaded.flows, ["flow_id"]),
    check_dtype=False,
    check_categorical=False,
)

pd.testing.assert_frame_equal(
    sort_df(flows.flow_to_trips, ["flow_id", "movement_id"]),
    sort_df(flows_loaded.flow_to_trips, ["flow_id", "movement_id"]),
    check_dtype=False,
    check_categorical=False,
)

assert flows_loaded.aggregation_spec == flows.aggregation_spec
assert flows_loaded.provenance == flows.provenance
assert flows_loaded.metadata.get("dataset_id") == flows.metadata.get("dataset_id")
assert flows_loaded.metadata.get("artifact_id") == flows.metadata.get("artifact_id")
assert flows_loaded.source_trips is None
assert flows_loaded.metadata.get("is_validated") is False

print("Comparación final: OK")
print("dataset_id =", flows_loaded.metadata.get("dataset_id"))
print("artifact_id =", flows_loaded.metadata.get("artifact_id"))

Comparación final: OK
dataset_id = flows_27dedf8e1df4
artifact_id = art_f2350103-946d-49f9-9328-f1760ad597d2


## 9. Cierre

Con esta demo dejé probado un happy path completo del bloque:

`Import trips → Validate trips → Build flows → Write flows → Read flows`

Lo importante aquí era confirmar que:
- el import deja un `TripDataset` rico y bien estructurado,
- la validación formal pasa,
- la construcción de flows entrega una tabla útil,
- la persistencia formal funciona,
- y el artefacto leído se puede seguir usando en el pipeline.